In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.mlflow_setup import init_tracking
from src.features import build_features
from src.data_split import time_based_split, wmae
import mlflow

init_tracking()

train = pd.read_csv('../data/train.csv')
features = pd.read_csv('../data/features.csv')
stores = pd.read_csv('../data/stores.csv')

df = build_features(train, features, stores)

df = df.sort_values(['Store', 'Dept', 'Date'])
df['Sales_Lag_1'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
df['Sales_Lag_52'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
df['Sales_Lag_1'] = df['Sales_Lag_1'].fillna(df['Sales_Lag_1'].median())
df['Sales_Lag_52'] = df['Sales_Lag_52'].fillna(df['Sales_Lag_52'].median())

train_set, val_set = time_based_split(df)
feature_cols = [c for c in train_set.columns if c not in ['Weekly_Sales', 'Date']]
X_train, y_train = train_set[feature_cols], train_set['Weekly_Sales']
X_val, y_val = val_set[feature_cols], val_set['Weekly_Sales']
val_is_holiday = val_set['IsHoliday']

print(train_set.shape, val_set.shape)

C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(391919, 24) (29651, 24)


In [2]:
!pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 2.3 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.9 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.7 MB/s  0:00:00


In [3]:
import lightgbm as lgb

mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_baseline"):
    params = {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05, 'random_state': 42}
    mlflow.log_params(params)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    score = wmae(y_val.values, preds, val_is_holiday.values)

    mlflow.log_metric("val_wmae", score)
    mlflow.lightgbm.log_model(model, "model")
    print(f"Validation WMAE: {score:.2f}")

2026/07/12 21:11:46 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_Training' does not exist. Creating a new experiment.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021677 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3299
[LightGBM] [Info] Number of data points in the train set: 391919, number of used features: 22
[LightGBM] [Info] Start training from score 16017.607716


2026/07/12 21:11:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Validation WMAE: 1405.33
🏃 View run LightGBM_baseline at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3/runs/44657abad5d24d84a958f83822bbb058
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3


In [4]:
param_options = [
    {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.03},
    {'n_estimators': 300, 'max_depth': 6,  'learning_rate': 0.1},
    {'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.02, 'num_leaves': 60},  # -1 = no limit, common for LightGBM
]

for i, params in enumerate(param_options):
    with mlflow.start_run(run_name=f"LightGBM_HPO_{i+1}"):
        mlflow.log_params(params)
        m = lgb.LGBMRegressor(**params, random_state=42)
        m.fit(X_train, y_train)
        preds = m.predict(X_val)
        score = wmae(y_val.values, preds, val_is_holiday.values)
        mlflow.log_metric("val_wmae", score)
        mlflow.lightgbm.log_model(m, "model")
        print(f"Run {i+1}: WMAE = {score:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009017 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3299
[LightGBM] [Info] Number of data points in the train set: 391919, number of used features: 22
[LightGBM] [Info] Start training from score 16017.607716


2026/07/12 21:14:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 1: WMAE = 1400.62
🏃 View run LightGBM_HPO_1 at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3/runs/9b5154b5d89e4608a6cd859705f65106
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010054 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3299
[LightGBM] [Info] Number of data points in the train set: 391919, number of used features: 22
[LightGBM] [Info] Start training from score 16017.607716
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

2026/07/12 21:15:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 2: WMAE = 1415.44
🏃 View run LightGBM_HPO_2 at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3/runs/36e719b152e5411cbe797bd3a2551cce
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014946 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3299
[LightGBM] [Info] Number of data points in the train set: 391919, number of used features: 22
[LightGBM] [Info] Start training from score 16017.607716


2026/07/12 21:15:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 3: WMAE = 1353.91
🏃 View run LightGBM_HPO_3 at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3/runs/5c2e420c2c154e4dad0d21277af0ae73
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, features_df, stores_df):
        self.features_df = features_df
        self.stores_df = stores_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        from src.features import build_features
        df = build_features(X, self.features_df, self.stores_df)
        df = df.sort_values(['Store', 'Dept', 'Date'])
        if 'Weekly_Sales' in df.columns:
            df['Sales_Lag_1'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
            df['Sales_Lag_52'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
        else:
            df['Sales_Lag_1'] = 0
            df['Sales_Lag_52'] = 0
        df['Sales_Lag_1'] = df['Sales_Lag_1'].fillna(0)
        df['Sales_Lag_52'] = df['Sales_Lag_52'].fillna(0)
        drop_cols = [c for c in ['Weekly_Sales', 'Date'] if c in df.columns]
        return df.drop(columns=drop_cols)

best_params = {'n_estimators': 800, 'max_depth': -1, 'learning_rate': 0.02, 'num_leaves': 60, 'random_state': 42}

pipeline = Pipeline([
    ('features', FeatureBuilder(features, stores)),
    ('model', lgb.LGBMRegressor(**best_params))
])

y_full = train.merge(features, on=['Store','Date','IsHoliday'], how='left')['Weekly_Sales']
pipeline.fit(train, y_full)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015236 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3295
[LightGBM] [Info] Number of data points in the train set: 421570, number of used features: 22
[LightGBM] [Info] Start training from score 15981.258121


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,features_df,Store ... x 12 columns]
,stores_df,Store Typ...5 B 118221
,num_leaves,60
,learning_rate,0.02
,n_estimators,800
,random_state,42
,boosting_type,'gbdt'


In [6]:
mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_final_pipeline"):
    mlflow.log_params(best_params)
    mlflow.log_metric("val_wmae", 1353.91)
    mlflow.sklearn.log_model(
        pipeline,
        "model",
        registered_model_name="walmart_lightgbm_model",
        serialization_format="cloudpickle"
    )
    print("LightGBM pipeline registered")

2026/07/12 21:18:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 21:18:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'walmart_lightgbm_model'.
2026/07/12 21:18:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart_lightgbm_model, version 1
Created version '1' of model 'walmart_lightgbm_model'.


LightGBM pipeline registered
🏃 View run LightGBM_final_pipeline at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3/runs/54236332348c4b3d8794b348afa3df9a
🧪 View experiment at: https://dagshub.com/ekali-star/walmart-recruiting.mlflow/#/experiments/3
